# 02 — Data cleaning

Apply the cleaning decisions documented in `01_data_audit.ipynb`. **Target remains** `construction_end_date - construction_start_date` (calendar days). Inspection is never used as the target.

**Read-only contract:** `data/build_history.csv` and `data/homes_to_predict.csv` are not opened for write. All outputs go under `data/processed/`.

**Rule for every fix:** re-count the issue on the raw extract first. If the issue is gone, skip that fix and log it. If it is still present, apply the approved treatment.

This notebook does not impute `beds`, `lot_size_sqft`, or `sale_date`; does not drop 2025 completes for truncation; and does not engineer model features (permit lags, missingness flags as features, etc.). Those were deferred in the audit.


## Approved vs deferred

| Issue | Action in this notebook |
|---|---|
| Exact duplicate rows | Drop extra copies; keep one row per `home_id` |
| `basement_type` `"None"` | Recode to `No basement` (do not impute) |
| `garage_size` words vs digits | Map Single→1, Double→2, Triple→3 |
| `community` case / whitespace | Strip + Title Case |
| Extra-zero `sqft` | Divide by 10 when the result falls in that plan's typical range |
| End date before start (7 homes) | Do not rewrite dates. Exclude from the training file |
| In-progress history | Do not impute an end date. Exclude from the training file |
| Target | `cycle_days` from end − start only |
| Snapshot truncation | Document only; keep 2025 completes that have a valid target |
| Missing `sale_date` / `beds` / `lot_size_sqft` | Verify; leave missing |
| Mix shift, `selected_options_value`, `site_manager` | Verify / keep as-is (no transform) |


In [1]:
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 80)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

ROOT = Path.cwd()
RAW_DIR = ROOT / "data"
OUT_DIR = RAW_DIR / "processed"
HIST_RAW = RAW_DIR / "build_history.csv"
PRED_RAW = RAW_DIR / "homes_to_predict.csv"
RAW_FILES = (HIST_RAW, PRED_RAW)

assert HIST_RAW.exists() and PRED_RAW.exists(), "Run from the project root."
OUT_DIR.mkdir(parents=True, exist_ok=True)


def file_sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def assert_write_allowed(path: Path) -> None:
    path = path.resolve()
    out = OUT_DIR.resolve()
    if path.parent != out:
        raise RuntimeError(f"Refusing to write outside {out}: {path}")
    if path in {p.resolve() for p in RAW_FILES}:
        raise RuntimeError(f"Refusing to overwrite a raw extract: {path}")


def safe_to_csv(df: pd.DataFrame, path: Path) -> None:
    assert_write_allowed(path)
    df.to_csv(path, index=False)
    print("wrote", path.relative_to(ROOT), "rows", len(df), "cols", df.shape[1])


RAW_HASH_BEFORE = {p.name: file_sha256(p) for p in RAW_FILES}

# Preserve the literal token None on basement_type.
hist = pd.read_csv(HIST_RAW, keep_default_na=False, na_values=[""])
pred = pd.read_csv(PRED_RAW, keep_default_na=False, na_values=[""])

print("history", hist.shape, "| predict", pred.shape)
print("raw SHA-256 at load:")
for name, digest in RAW_HASH_BEFORE.items():
    print(f"  {name}: {digest}")


history (4684, 30) | predict (578, 30)
raw SHA-256 at load:
  build_history.csv: 1faf14257bc593cacc7933c8ac43891adec57169728413aa17007f9703b7f340
  homes_to_predict.csv: 362c05ce1ab9eac915c66c7e0b4a00bcd80acdb00b171d53e42ef4892448df24


## Helpers

`require_issue` re-counts the defect. The approved fix runs only when the count is greater than zero.


In [2]:
log_rows = []


def log_issue(**kwargs):
    log_rows.append(kwargs)


def require_issue(name, n_found, n_expected=None) -> bool:
    msg = f"{name}: found {n_found}"
    if n_expected is not None:
        msg += f" (audit {n_expected})"
    print(msg)
    if n_found == 0:
        print("  SKIP — issue not present in current raw data")
        log_issue(issue=name, n_found=n_found, n_expected=n_expected, applied=False, action="skipped_not_found")
        return False
    if n_expected is not None and n_found != n_expected:
        print("  WARN — count differs from audit; applying because the issue still exists")
    return True


GARAGE_MAP = {"Single": 1, "Double": 2, "Triple": 3, "1": 1, "2": 2, "3": 3, 1: 1, 2: 2, 3: 3}
DATE_COLS = [
    "sale_date", "permit_application_date", "permit_issue_date",
    "construction_start_date", "construction_end_date", "final_inspection_date",
]
SNAPSHOT_CUT = pd.Timestamp("2025-07-31")

# Working copies. Issue counts below always use hist/pred (raw), never a partially cleaned frame.
hist_c = hist.copy()
pred_c = pred.copy()

def as_bool(s):
    if pd.api.types.is_bool_dtype(s):
        return s.astype(bool)
    return s.astype(str).str.strip().str.lower().isin(["true", "1", "yes"])

hist["is_spec_home"] = as_bool(hist["is_spec_home"])
pred["is_spec_home"] = as_bool(pred["is_spec_home"])
hist_c["is_spec_home"] = as_bool(hist_c["is_spec_home"])
pred_c["is_spec_home"] = as_bool(pred_c["is_spec_home"])

raw_end = pd.to_datetime(hist["construction_end_date"], errors="coerce")
raw_start = pd.to_datetime(hist["construction_start_date"], errors="coerce")
RAW_COUNTS = {
    "dup_hist": int(hist.duplicated().sum()),
    "dup_pred": int(pred.duplicated().sum()),
    "basement_none_hist": int((hist.basement_type.astype(str) == "None").sum()),
    "basement_none_pred": int((pred.basement_type.astype(str) == "None").sum()),
    "garage_words_hist": int(hist.garage_size.astype(str).isin(["Single", "Double", "Triple"]).sum()),
    "garage_words_pred": int(pred.garage_size.astype(str).isin(["Single", "Double", "Triple"]).sum()),
    "community_dirty_hist": int((hist.community.astype(str) != hist.community.astype(str).str.strip().str.title()).sum()),
    "community_dirty_pred": int((pred.community.astype(str) != pred.community.astype(str).str.strip().str.title()).sum()),
    "community_raw_nunique_hist": int(hist.community.nunique()),
    "community_raw_nunique_pred": int(pred.community.nunique()),
    "sqft_out_hist": int((hist.sqft > 10_000).sum()),
    "sqft_out_pred": int((pred.sqft > 10_000).sum()),
    "end_before_start": int((raw_end.notna() & (raw_end < raw_start)).sum()),
    "in_progress": int(hist.construction_status.eq("In Progress").sum()),
    "beds_miss_hist": int(hist.beds.isna().sum()),
    "beds_miss_pred": int(pred.beds.isna().sum()),
    "lot_miss_hist": int(hist.lot_size_sqft.isna().sum()),
    "lot_miss_pred": int(pred.lot_size_sqft.isna().sum()),
    "nonspec_sale_miss_hist": int(((~hist.is_spec_home.astype(bool)) & hist.sale_date.isna()).sum()),
    "nonspec_sale_miss_pred": int(((~pred.is_spec_home.astype(bool)) & pred.sale_date.isna()).sum()),
    "spec_sale_miss_hist": int((hist.is_spec_home.astype(bool) & hist.sale_date.isna()).sum()),
    "spec_sale_miss_pred": int((pred.is_spec_home.astype(bool) & pred.sale_date.isna()).sum()),
}
print("raw issue counts (before any fix):")
display(pd.Series(RAW_COUNTS, name="n").to_frame())


raw issue counts (before any fix):


,n
dup_hist,25
dup_pred,0
basement_none_hist,1654
basement_none_pred,260
garage_words_hist,259
garage_words_pred,31
community_dirty_hist,398
community_dirty_pred,49
community_raw_nunique_hist,70
community_raw_nunique_pred,45


## 1. Exact duplicate rows (history)

Approved: keep one row per `home_id`. Copies are identical, so `drop_duplicates` is enough. Scoring file had none in the audit.


In [3]:
print("predict exact duplicate extras (raw):", RAW_COUNTS["dup_pred"])

if require_issue("exact duplicate extras (history)", RAW_COUNTS["dup_hist"], 25):
    before = len(hist_c)
    hist_c = hist_c.drop_duplicates().reset_index(drop=True)
    log_issue(
        issue="exact duplicate extras (history)",
        n_found=RAW_COUNTS["dup_hist"], n_expected=25, applied=True,
        action=f"drop_duplicates: {before} -> {len(hist_c)} rows; unique home_id={hist_c.home_id.nunique()}",
    )
    print("  applied: rows now", len(hist_c), "unique home_id", hist_c.home_id.nunique())

assert hist_c.home_id.nunique() == len(hist_c), "home_id is not unique after de-duplication"
assert pred_c.home_id.nunique() == len(pred_c), "predict home_id is not unique"


predict exact duplicate extras (raw): 0
exact duplicate extras (history): found 25 (audit 25)
  applied: rows now 4659 unique home_id 4659


## 2. `basement_type` token `None`

Approved: this is a real level (no basement), not missing. Recode to `No basement` so a later default `read_csv` cannot swallow it.


In [4]:
for label, df, n, expected in [
    ("history", hist_c, RAW_COUNTS["basement_none_hist"], 1654),
    ("predict", pred_c, RAW_COUNTS["basement_none_pred"], 260),
]:
    if require_issue(f"basement_type == 'None' ({label})", n, expected):
        df["basement_type"] = df["basement_type"].replace({"None": "No basement"})
        log_issue(issue=f"basement_type None ({label})", n_found=n, n_expected=expected,
                  applied=True, action="recode None -> 'No basement'")

print("basement after recode:")
display(pd.DataFrame({
    "history": hist_c.basement_type.value_counts(dropna=False),
    "predict": pred_c.basement_type.value_counts(dropna=False),
}).fillna(0).astype(int))


basement_type == 'None' (history): found 1654 (audit 1654)
basement_type == 'None' (predict): found 260 (audit 260)
basement after recode:


,history,predict
basement_type,,
Finished,972,103
No basement,1644,260
Unfinished,2043,215


## 3. `garage_size` mixed words and digits

Approved: Single→1, Double→2, Triple→3. Store as integer.


In [5]:
for label, df, n, expected in [
    ("history", hist_c, RAW_COUNTS["garage_words_hist"], 259),
    ("predict", pred_c, RAW_COUNTS["garage_words_pred"], 31),
]:
    if require_issue(f"garage_size word tokens ({label})", n, expected):
        unknown = sorted(set(df.garage_size.astype(str).unique()) - set(map(str, GARAGE_MAP)))
        if unknown:
            raise ValueError(f"unmapped garage_size values in {label}: {unknown}")
        df["garage_size"] = df["garage_size"].map(lambda x: GARAGE_MAP[int(x) if str(x).isdigit() else x]).astype("int64")
        log_issue(issue=f"garage_size words ({label})", n_found=n, n_expected=expected,
                  applied=True, action="map Single/Double/Triple -> 1/2/3")

print("garage_size after map:")
display(pd.DataFrame({
    "history": hist_c.garage_size.value_counts(dropna=False),
    "predict": pred_c.garage_size.value_counts(dropna=False),
}).fillna(0).astype(int).sort_index())


garage_size word tokens (history): found 259 (audit 259)
garage_size word tokens (predict): found 31 (audit 31)
garage_size after map:


,history,predict
garage_size,,
1,1359,242
2,1861,206
3,1439,130


## 4. `community` capitalization and whitespace

Approved: strip + one canonical Title Case. Do not drop rows.


In [6]:
for label, df, dirty, raw_n, expected_raw_nunique in [
    ("history", hist_c, RAW_COUNTS["community_dirty_hist"], RAW_COUNTS["community_raw_nunique_hist"], 70),
    ("predict", pred_c, RAW_COUNTS["community_dirty_pred"], RAW_COUNTS["community_raw_nunique_pred"], 45),
]:
    print(f"{label} raw unique communities: {raw_n} (audit {expected_raw_nunique})")
    if require_issue(f"community case/whitespace rows ({label})", dirty):
        df["community"] = df["community"].astype(str).str.strip().str.title()
        log_issue(issue=f"community labels ({label})", n_found=dirty, n_expected=None,
                  applied=True, action=f"strip+title: unique {raw_n} -> {df.community.nunique()}")

print("canonical communities:")
display(pd.DataFrame({
    "history": hist_c.community.value_counts(),
    "predict": pred_c.community.value_counts(),
}).fillna(0).astype(int).sort_index())
assert hist_c.community.nunique() == 14
assert set(pred_c.community) <= set(hist_c.community)


history raw unique communities: 70 (audit 70)
community case/whitespace rows (history): found 398
predict raw unique communities: 45 (audit 45)
community case/whitespace rows (predict): found 49
canonical communities:


,history,predict
community,,
Auburn Meadows,328,56
Aurora Highlands,344,35
Barefoot Lakes,323,44
Cranston Ridge,294,39
Glenridding,360,40
Green Valley,317,38
Keswick Landing,335,35
Laurel Green,348,48
Legacy Gate,331,44


## 5. Extra-zero `sqft`

Approved: if `sqft > 10,000` and `sqft / 10` falls inside that plan's typical range (min–max of history rows with `sqft ≤ 10,000`), divide by 10. Apply the same rule to both files. Leave the value unchanged if the guard fails.


In [7]:
plan_range = (
    hist_c.loc[hist_c.sqft <= 10_000]
    .groupby("plan_name")["sqft"]
    .agg(typical_min="min", typical_max="max", typical_median="median")
)

def sqft_fix_table(df, label):
    t = df.loc[df.sqft > 10_000, ["home_id", "plan_name", "sqft"]].copy()
    t = t.merge(plan_range, left_on="plan_name", right_index=True, how="left")
    t["sqft_div10"] = t.sqft / 10
    t["in_typical_range"] = (t.sqft_div10 >= t.typical_min) & (t.sqft_div10 <= t.typical_max)
    t["dataset"] = label
    return t

outliers = pd.concat([sqft_fix_table(hist_c, "history"), sqft_fix_table(pred_c, "predict")], ignore_index=True)
print("sqft > 10,000 before fix:")
display(outliers.sort_values(["dataset", "sqft"], ascending=[True, False]))

for label, df, n, expected in [
    ("history", hist_c, RAW_COUNTS["sqft_out_hist"], 13),
    ("predict", pred_c, RAW_COUNTS["sqft_out_pred"], 3),
]:
    if not require_issue(f"sqft > 10000 ({label})", n, expected):
        continue
    sub = outliers[outliers.dataset.eq(label)]
    apply_ids = sub.loc[sub.in_typical_range, "home_id"]
    skip_ids = sub.loc[~sub.in_typical_range, "home_id"]
    df.loc[df.home_id.isin(apply_ids), "sqft"] = (df.loc[df.home_id.isin(apply_ids), "sqft"] / 10).round().astype("int64")
    log_issue(
        issue=f"sqft extra zero ({label})",
        n_found=n, n_expected=expected, applied=True,
        action=f"divided {len(apply_ids)} by 10; left unchanged {list(skip_ids) if len(skip_ids) else 'none'}",
    )
    print(f"  applied ÷10 to {len(apply_ids)} {label} rows; unchanged: {list(skip_ids) if len(skip_ids) else 'none'}")

print("sqft > 10,000 after fix  hist/pred:", int((hist_c.sqft > 10_000).sum()), int((pred_c.sqft > 10_000).sum()))


sqft > 10,000 before fix:


,home_id,plan_name,sqft,typical_min,typical_max,typical_median,sqft_div10,in_typical_range,dataset
1,H100569,Nyssa,34900,3050,3830,"3,410.00","3,490.00",True,history
3,H102804,Larch,30800,2520,3190,"2,860.00","3,080.00",True,history
9,H100489,Larch,30600,2520,3190,"2,860.00","3,060.00",True,history
5,H100450,Maple,30500,2740,3490,"3,090.00","3,050.00",True,history
11,H104295,Larch,28400,2520,3190,"2,860.00","2,840.00",True,history
10,H104499,Larch,27400,2520,3190,"2,860.00","2,740.00",True,history
0,H102544,Katsura,26200,2250,2880,"2,610.00","2,620.00",True,history
2,H100767,Oakmont,23100,1920,2460,"2,195.00","2,310.00",True,history
4,H101916,Oakmont,22000,1920,2460,"2,195.00","2,200.00",True,history
12,H100969,Hawthorn,19400,1610,2070,"1,840.00","1,940.00",True,history


sqft > 10000 (history): found 13 (audit 13)
  applied ÷10 to 13 history rows; unchanged: none
sqft > 10000 (predict): found 3 (audit 3)
  applied ÷10 to 3 predict rows; unchanged: none
sqft > 10,000 after fix  hist/pred: 0 0


## 6. Dates, target, inverted ends, in-progress

Parse dates explicitly (audit: all non-empty values were already ISO).

**Target (fixed):** `cycle_days = construction_end_date - construction_start_date`. Do not rewrite any date. Rows with a missing or non-positive cycle are excluded from the training file only.


In [8]:
def parse_dates(df):
    out = df.copy()
    for c in DATE_COLS:
        out[c] = pd.to_datetime(out[c], errors="coerce")
    return out

hist_c = parse_dates(hist_c)
pred_c = parse_dates(pred_c)

# Re-verify inverted ends and in-progress on the cleaned copies (post-dedup).
end_before_start = hist_c.construction_end_date.notna() & (
    hist_c.construction_end_date < hist_c.construction_start_date
)
inverted_ids = hist_c.loc[end_before_start, "home_id"].tolist()
print("inverted end < start home_ids (after de-dup):", inverted_ids)

if require_issue("end date before start", RAW_COUNTS["end_before_start"], 7):
    log_issue(
        issue="end date before start",
        n_found=RAW_COUNTS["end_before_start"], n_expected=7, applied=True,
        action="no date rewrite; exclude from training file: " + ",".join(inverted_ids),
    )

inprog = hist_c.construction_status.eq("In Progress")
if require_issue("in-progress history (no target)", RAW_COUNTS["in_progress"], 579):
    assert hist_c.loc[inprog, "construction_end_date"].isna().all()
    log_issue(
        issue="in-progress history",
        n_found=RAW_COUNTS["in_progress"], n_expected=579, applied=True,
        action=f"no end-date imputation; exclude {int(inprog.sum())} remaining in-progress rows from training file",
    )

hist_c["cycle_days"] = (hist_c.construction_end_date - hist_c.construction_start_date).dt.days
# Scoring has no end date — do not invent a target.
pred_c["cycle_days"] = pd.NA

valid_cycle = hist_c.cycle_days.notna() & (hist_c.cycle_days > 0)
hist_c["train_eligible"] = hist_c.construction_status.eq("Complete") & valid_cycle
hist_c["exclude_reason"] = np.where(
    hist_c.train_eligible, "",
    np.where(inprog, "in_progress",
             np.where(end_before_start, "end_before_start",
                      np.where(hist_c.construction_end_date.isna(), "missing_end_date", "nonpositive_cycle"))),
)
pred_c["train_eligible"] = False
pred_c["exclude_reason"] = "scoring_set"

print("history exclude_reason:")
display(hist_c.exclude_reason.value_counts().to_frame("n"))
print("cycle_days on train-eligible rows:")
display(hist_c.loc[hist_c.train_eligible, "cycle_days"].describe().to_frame("cycle_days"))
assert (hist_c.loc[hist_c.train_eligible, "cycle_days"] > 0).all()
assert hist_c.loc[hist_c.train_eligible, "construction_status"].eq("Complete").all()

print("snapshot documentation (no rows dropped for truncation):")
print("  history start max:", hist_c.construction_start_date.max().date(),
      "| history end max:", hist_c.construction_end_date.max().date(),
      "| predict start min:", pred_c.construction_start_date.min().date())
print("  inferred cut:", SNAPSHOT_CUT.date(), "— 2025 completes with a valid target are kept")
log_issue(issue="snapshot truncation", n_found=int((hist_c.construction_start_date.dt.year == 2025).sum()),
          n_expected=None, applied=False,
          action="documented cut 2025-07-31; 2025 completes with valid cycle_days kept")


inverted end < start home_ids (after de-dup): ['H100392', 'H103515', 'H103849', 'H103689', 'H101045', 'H101204', 'H101439']
end date before start: found 7 (audit 7)
in-progress history (no target): found 579 (audit 579)
history exclude_reason:


,n
exclude_reason,
,4077
in_progress,575
end_before_start,7


cycle_days on train-eligible rows:


,cycle_days
count,"4,077.00"
mean,187.54
std,62.64
min,65.00
25%,140.00
50%,178.00
75%,224.00
max,622.00


snapshot documentation (no rows dropped for truncation):
  history start max: 2025-07-31 | history end max: 2025-07-31 | predict start min: 2025-08-01
  inferred cut: 2025-07-31 — 2025 completes with a valid target are kept


## 7. Deferred issues — verify only

These defects are still in the extracts. The audit said not to impute or drop for them yet, so this notebook only re-counts them.


In [9]:
deferred = pd.DataFrame([
    ["beds missing (history)", RAW_COUNTS["beds_miss_hist"], 141, "leave missing; do not global-mean impute"],
    ["beds missing (predict)", RAW_COUNTS["beds_miss_pred"], 17, "leave missing"],
    ["lot_size_sqft missing (history)", RAW_COUNTS["lot_miss_hist"], 97, "leave missing"],
    ["lot_size_sqft missing (predict)", RAW_COUNTS["lot_miss_pred"], 12, "leave missing"],
    ["non-spec missing sale_date (history)", RAW_COUNTS["nonspec_sale_miss_hist"], 504,
     "leave missing; spec missingness is expected and separate"],
    ["non-spec missing sale_date (predict)", RAW_COUNTS["nonspec_sale_miss_pred"], 62, "leave missing"],
    ["spec missing sale_date (history)", RAW_COUNTS["spec_sale_miss_hist"], 1081, "expected; leave as NA"],
    ["spec missing sale_date (predict)", RAW_COUNTS["spec_sale_miss_pred"], 134, "expected; leave as NA"],
])
deferred.columns = ["issue", "n_found", "n_audit", "decision"]
display(deferred)

for _, r in deferred.iterrows():
    still = require_issue(r.issue, int(r.n_found), int(r.n_audit))
    log_issue(issue=r.issue, n_found=int(r.n_found), n_expected=int(r.n_audit),
              applied=False, action=("verified_leave_as_is" if still else "skipped_not_found") + f" | {r.decision}")

print("spec with a sale_date (should be 0):",
      int((hist_c.is_spec_home & hist_c.sale_date.notna()).sum()),
      int((pred_c.is_spec_home & pred_c.sale_date.notna()).sum()))


,issue,n_found,n_audit,decision
0,beds missing (history),141,141,leave missing; do not global-mean impute
1,beds missing (predict),17,17,leave missing
2,lot_size_sqft missing (history),97,97,leave missing
3,lot_size_sqft missing (predict),12,12,leave missing
4,non-spec missing sale_date (history),504,504,leave missing; spec missingness is expected and separate
5,non-spec missing sale_date (predict),62,62,leave missing
6,spec missing sale_date (history),1081,1081,expected; leave as NA
7,spec missing sale_date (predict),134,134,expected; leave as NA


beds missing (history): found 141 (audit 141)
beds missing (predict): found 17 (audit 17)
lot_size_sqft missing (history): found 97 (audit 97)
lot_size_sqft missing (predict): found 12 (audit 12)
non-spec missing sale_date (history): found 504 (audit 504)
non-spec missing sale_date (predict): found 62 (audit 62)
spec missing sale_date (history): found 1081 (audit 1081)
spec missing sale_date (predict): found 134 (audit 134)
spec with a sale_date (should be 0): 0 0


## 8. Post-clean checks, then write `data/processed` only

`build_history_cleaned.csv` / `homes_to_predict_cleaned.csv` are value-cleaned copies of each extract (history de-duplicated). `build_history_train.csv` is the supervised subset: Complete, `cycle_days > 0`. Dates are written as `YYYY-MM-DD`.


In [10]:
def dates_to_iso(df):
    out = df.copy()
    for c in DATE_COLS:
        out[c] = out[c].dt.strftime("%Y-%m-%d").where(out[c].notna(), "")
    return out


print("post-clean guards")
print("  hist home_id unique:", hist_c.home_id.is_unique, "pred:", pred_c.home_id.is_unique)
print("  garage values:", sorted(hist_c.garage_size.unique().tolist()), sorted(pred_c.garage_size.unique().tolist()))
print("  basement values hist:", sorted(hist_c.basement_type.dropna().unique().tolist()))
print("  community nunique hist/pred:", hist_c.community.nunique(), pred_c.community.nunique())
print("  sqft max hist/pred:", int(hist_c.sqft.max()), int(pred_c.sqft.max()))
print("  train rows:", int(hist_c.train_eligible.sum()),
      "in-progress remaining in cleaned history:", int((hist_c.exclude_reason == "in_progress").sum()),
      "end_before_start remaining in cleaned history:", int((hist_c.exclude_reason == "end_before_start").sum()))

train = hist_c.loc[hist_c.train_eligible].copy()
assert train.cycle_days.gt(0).all()
assert train.construction_end_date.notna().all()
assert train.home_id.is_unique

hist_out = dates_to_iso(hist_c)
pred_out = dates_to_iso(pred_c)
train_out = dates_to_iso(train)
# cycle_days is numeric; empty string dates only on the date columns.
hist_out["cycle_days"] = hist_c.cycle_days
pred_out["cycle_days"] = pd.NA
train_out["cycle_days"] = train.cycle_days.astype("int64")

safe_to_csv(hist_out, OUT_DIR / "build_history_cleaned.csv")
safe_to_csv(pred_out, OUT_DIR / "homes_to_predict_cleaned.csv")
safe_to_csv(train_out, OUT_DIR / "build_history_train.csv")

log_df = pd.DataFrame(log_rows)
safe_to_csv(log_df, OUT_DIR / "cleaning_log.csv")
print("\\ncleaning log:")
display(log_df)

raw_hash_after = {p.name: file_sha256(p) for p in RAW_FILES}
print("\\nraw SHA-256 after writes (must match load):")
for name in RAW_HASH_BEFORE:
    same = RAW_HASH_BEFORE[name] == raw_hash_after[name]
    print(f"  {name}: {'UNCHANGED' if same else 'CHANGED'}")
    assert same, f"Raw file was modified: {name}"

print("processed outputs:", sorted(p.name for p in OUT_DIR.glob('*.csv')))


post-clean guards
  hist home_id unique: True pred: True
  garage values: [1, 2, 3] [1, 2, 3]
  basement values hist: ['Finished', 'No basement', 'Unfinished']
  community nunique hist/pred: 14 14
  sqft max hist/pred: 3830 3610
  train rows: 4077 in-progress remaining in cleaned history: 575 end_before_start remaining in cleaned history: 7
wrote data\processed\build_history_cleaned.csv rows 4659 cols 33
wrote data\processed\homes_to_predict_cleaned.csv rows 578 cols 33


wrote

 data\processed\build_history_train.csv rows 4077 cols 33


wrote data\processed\cleaning_log.csv rows 20 cols 5
\ncleaning log:


,issue,n_found,n_expected,applied,action
0,exact duplicate extras (history),25,25.00,True,drop_duplicates: 4684 -> 4659 rows; unique home_id=4659
1,basement_type None (history),1654,"1,654.00",True,recode None -> 'No basement'
2,basement_type None (predict),260,260.00,True,recode None -> 'No basement'
3,garage_size words (history),259,259.00,True,map Single/Double/Triple -> 1/2/3
4,garage_size words (predict),31,31.00,True,map Single/Double/Triple -> 1/2/3
5,community labels (history),398,NaN,True,strip+title: unique 70 -> 14
6,community labels (predict),49,NaN,True,strip+title: unique 45 -> 14
7,sqft extra zero (history),13,13.00,True,divided 13 by 10; left unchanged none
8,sqft extra zero (predict),3,3.00,True,divided 3 by 10; left unchanged none
9,end date before start,7,7.00,True,"no date rewrite; exclude from training file: H100392,H103515,H103849,H103689,H101045,H101204,H10..."


\nraw SHA-256 after writes (must match load):
  build_history.csv: UNCHANGED
  homes_to_predict.csv: UNCHANGED
processed outputs: ['build_history_cleaned.csv', 'build_history_train.csv', 'cleaning_log.csv', 'homes_to_predict_cleaned.csv']


## 9. What was produced

| File | Contents |
|---|---|
| `data/processed/build_history_cleaned.csv` | History after de-duplication and value cleaning. Includes in-progress and the 7 inverted-end rows, with `cycle_days`, `train_eligible`, `exclude_reason`. |
| `data/processed/homes_to_predict_cleaned.csv` | Scoring file after the same value cleaning. No target. |
| `data/processed/build_history_train.csv` | Complete homes with `cycle_days > 0` only. Use this for later modeling. |
| `data/processed/cleaning_log.csv` | Each issue: count found, whether the fix ran, action taken. |

Raw extracts under `data/` are unchanged. Next stages should read the processed files, not the raw CSVs.
